In [1]:
from skidl.logger import stop_log_file_output
stop_log_file_output(True)

In [2]:
from python.spice_tools import search_spice_model, save_part_model

entry = search_spice_model(name="FDMC510P-MS", library="Triode_MOS_Tube_Transistor")
if entry:
    print(entry["model_content"])
else:
    print("Not in model db")
    ## Save part model
    # save_part_model(
    # name="FDMC510P-MS",
    # library="Device",
    # model_content=".MODEL D1 D\n",
    # vendor_provided=True,
    # )


* FDMC510P-MS P-channel MOSFET (DFN3x3-8)
* Pin order: S1 S2 S3 G D
.SUBCKT FDMC510P_MS S1 S2 S3 G D
RS1 S1 S 0.2m
RS2 S2 S 0.2m
RS3 S3 S 0.2m
RDRAIN D D_INT 0.5m
CGS G S 3n
CGD G D_INT 1n
MMAIN D_INT S G S FDMC510P_PMOS L=1u W=2000u
.MODEL FDMC510P_PMOS PMOS (LEVEL=1 VTO=-1.8 KP=60 RD=0.002 RS=0.002)
.ENDS FDMC510P_MS



In [3]:
# from pathlib import Path
# from python.spice_tools import convert_skidl_module

# convert_skidl_module(
#     input_path=Path("test_cases/case_3A_charger/skidl/modules/battery_protection.py"),
#     subckt_name="Battery_Protection",
#     output_path=Path("test_cases/case_3A_charger/spice/battery_protection/battery_protection_pyspice.py"),
# )

In [4]:
import json

test_bench_path = "test_cases/case_3A_charger/testbench/battery_protection_schema_valid.json"

with open(test_bench_path, "r") as f:
    test_bench = json.load(f)
    testcases = test_bench["use_cases"]
case_ids = [case["name"] for case in testcases]

case_ids

['startup_reverse_block_with_charger_ramp',
 'forward_conduction_drop_3A_25C',
 'forward_conduction_drop_3A_60C',
 'reverse_blocking_usb_removed',
 'overcharge_overvoltage_cutoff',
 'overdischarge_cutoff_under_load',
 'discharge_short_circuit_protection',
 'charge_direction_short_circuit_protection',
 'steady_ripple_pass_through']

In [5]:
from python.spice_tools.harness_sanity import harness_sanity_check

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    result = harness_sanity_check(harness_path=harness_path)
    print(result)


{'ok': True, 'max_abs_voltage': 3.179966143196661, 'max_abs_current': 15.600001559999889, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.9374998046875103, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.9374998046875103, 'max_abs_current': 0.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 2.924999878139598, 'max_abs_current': 19.50000243772368, 'num_points': 526, 'error': None}
{'ok': True, 'max_abs_voltage': 4.2851271236494615, 'max_abs_current': 0.0, 'num_points': 520, 'error': None}
{'ok': True, 'max_abs_voltage': 1.8333332722222246, 'max_abs_current': 3.0, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.8999999512500017, 'max_abs_current': 24.375100492934543, 'num_points': 523, 'error': None}
{'ok': True, 'max_abs_voltage': 4.124999795575323, 'max_abs_current': 1.5000040884935568, 'num_points': 508, 'error': None}
{'ok': True, 'max_abs_voltage': 3.0125132697252983, 'max_abs_cur

In [6]:
from python.spice_tools.testbench_runner import run_use_case

for idx in range(len(case_ids)):
    harness_path = f"test_cases/case_3A_charger/spice/battery_protection/testcases/{case_ids[idx]}.py"

    reports = run_use_case(
        schema_path=test_bench_path,
        harness_path=harness_path,
        use_case_name=case_ids[idx],
        dut_path="test_cases/case_3A_charger/spice/battery_protection/battery_protection_pyspice.py",
        dut_module_name="Battery_Protection_pyspice",
    )

    print(reports)


{'startup_reverse_block_with_charger_ramp': {'total_measurements': 3, 'num_passed': 3, 'all_passed': True, 'measurements': {'reverse_leakage_during_ramp': {'assertion': {'value': -1.1556215201815248e-06, 'op': '<=', 'limit': 0.001}, 'passed': True}, 'pack_overshoot_on_connect': {'assertion': {'value': 1.2458574349949458e-07, 'op': '<=', 'limit': 0.05}, 'passed': True}, 'pack_voltage_in_band': {'assertion': {'value': 1.0, 'op': '>=', 'limit': 0.99}, 'passed': True}}}}
{'forward_conduction_drop_3A_25C': {'total_measurements': 2, 'num_passed': 0, 'all_passed': False, 'measurements': {'vdrop_mean_3A_25C': {'assertion': {'value': 0.24999999987569013, 'op': '<=', 'limit': 0.12}, 'passed': False}, 'module_efficiency_3A_25C': {'assertion': {'value': 93.74999999998296, 'op': '>=', 'limit': 97.0}, 'passed': False}}}}
{'forward_conduction_drop_3A_60C': {'total_measurements': 2, 'num_passed': 0, 'all_passed': False, 'measurements': {'vdrop_mean_3A_60C': {'assertion': {'value': 0.24999999987562702,